# 9. 模型诊断：漂移监控、概率校准、不平衡重采样、KS/Lift/评分卡

本 notebook 演示 easymlops 的诊断扩展能力（本仓库源码新增，对应 `tests/test_diag_tools.py` 实测）：

| 能力 | 导入 |
|------|------|
| PSI/KS/Lift/评分卡刻度 | `easymlops.table.utils` |
| 类不平衡重采样 + 样本权重 | `resample` / `LGBMClassification(weight=)` |
| 概率校准 | `easymlops.table.calibration.IsotonicCalibrationPipe` |
| 漂移监控 | `easymlops.table.drift.DriftMonitorPipe` |

In [ ]:
import numpy as np
import pandas as pd
from easymlops.table.utils import calc_psi, calc_ks, calc_lift, prob2score

rng = np.random.RandomState(42)

# PSI: 同分布 <0.1 稳定; 0.1~0.25 需关注; >0.25 显著漂移
train_score = rng.normal(0, 1, 5000)
print('PSI 同分布 :', round(calc_psi(train_score, rng.normal(0, 1, 5000)), 4))
print('PSI 均值偏移:', round(calc_psi(train_score, rng.normal(2, 1, 5000)), 4))

# KS 区分度与对应阈值
y_true = (rng.rand(1000) < (1 / (1 + np.exp(-rng.normal(0, 1, 1000) * 2)))).astype(int)
y_score = y_true * 0.7 + rng.rand(1000) * 0.3
print('KS:', calc_ks(y_true, y_score))

# Lift 表（按分数降序等频分箱）与评分卡刻度
print(calc_lift(y_true, y_score, bins=5))
print('评分卡刻度:', np.round(prob2score(pd.Series([0.01, 0.05, 0.2, 0.5])), 1))

In [ ]:
# 类不平衡重采样：必须在构建 pipeline 之前调用（y 在模型 pipe 构造时绑定）
from easymlops.table.utils import resample

x_imb = pd.DataFrame({'a': rng.rand(500), 'b': rng.rand(500)})
y_imb = pd.Series([1] * 50 + [0] * 450)

x_over, y_over = resample(x_imb, y_imb, method='oversample')
x_under, y_under = resample(x_imb, y_imb, method='undersample')
print('过采样:', y_over.value_counts().to_dict())
print('欠采样:', y_under.value_counts().to_dict())
# method='smote' 需 pip install imblearn

In [ ]:
# 样本权重：weight 长度需与 y 一致
from easymlops.table.classification import LGBMClassification

x = pd.DataFrame({'a': rng.rand(1000), 'b': rng.rand(1000)})
y = ((x['a'] + x['b']) > 1.0).astype(int)

model = LGBMClassification(y=y, use_faster_predictor=False,
                           weight=np.where(y.values == 1, 2.0, 1.0),
                           native_fit_params={'num_boost_round': 10})
model.fit(x)
print(model.transform(x).head())

In [ ]:
# 漂移监控（透传 pipe）+ 概率校准（验证集无偏用法）
from easymlops import TablePipeLine
from easymlops.table.preprocessing import FixInput
from easymlops.table.drift import DriftMonitorPipe
from easymlops.table.calibration import IsotonicCalibrationPipe

x_train, x_val, x_test = x[:600], x[600:800], x[800:]
y_train, y_val, y_test = y[:600], y[600:800], y[800:]

table = TablePipeLine()
table.pipe(FixInput(reduce_mem_usage=False)) \
     .pipe(DriftMonitorPipe(psi_threshold=0.25)) \
     .pipe(LGBMClassification(y=y_train, use_faster_predictor=False,
                              native_fit_params={'num_boost_round': 10}))
table.fit(x_train)

# 校准 pipe 在验证集预测上单独 fit 后 append（推荐，无偏）
calib = IsotonicCalibrationPipe(y=y_val, cols=[1])
calib.fit(table.transform(x_val))
table.pipe(calib)

calibrated = table.transform(x_test)
print('校准后概率范围:', calibrated[1].min(), '~', calibrated[1].max())

In [ ]:
# 读取监控结果；对漂移数据再 transform 会命中阈值并记录
monitor = [m for m in table.models if isinstance(m, DriftMonitorPipe)][0]
print('正常数据 PSI:', {k: round(v, 4) for k, v in monitor.last_psi_report_.items()})

x_shift = x_test.copy()
x_shift['a'] = x_shift['a'] + 3.0   # 人造漂移
table.transform(x_shift)
print('漂移列:', monitor.last_drift_cols_)

# 模型分数也可监控：DriftMonitorPipe(cols=[正类列名]) 挂在模型 pipe 之后

## 注意事项

- pipe 的 `name` 默认是类对象而非字符串，检索 pipe 用 `[m for m in table.models if isinstance(m, XxxPipe)]`
- 校准/监控 pipe 的 y 必须与实际流入该 pipe 的数据行配对
- 仓库源码已修复 pandas<1.5 下 `copy_on_write` 设置导致的导入失败